# Imports

In [1]:
# Linear algebra
import numpy as np

# Dataframes
import pandas as pd

# Path
import os
data_path = os.path.join('..', 'data')

# Plotting
# import seaborn as sns
import matplotlib.pyplot as plt

# Pandas faster apply
import swifter

# Progress bar
from tqdm import tqdm

C:\Users\wsega\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Data

In [2]:
# Load test data
test_with_results = pd.read_pickle(os.path.join(data_path, '2_test_with_bart30.pkl'))

# Load SDA weights
SDAw_TCGA = pd.read_csv(os.path.join(data_path, 'SDA', 'tRNA_TCGA_Healthy', 'matched_AAcTE.csv'))

# Organize df to (codons x samples) shape
SDAw_TCGA.rename(columns={'Unnamed: 0': 'samples'}, inplace=True)
new_column_names = SDAw_TCGA['samples']
SDAw_TCGA = SDAw_TCGA.drop(columns=['samples']).T.rename(new_column_names, axis=1)
SDAw_TCGA.index.rename('Codon', inplace=True)
# Remove first 3 letters (amino acid) from index entries
SDAw_TCGA.index = SDAw_TCGA.index.str[3:]
# Drop NaN samples
SDAw_TCGA.dropna(axis=1, inplace=True)


In [3]:
SDAw_TCGA.head()

,COAD.TCGA.A6.2671.11A.01R.1757.13_mirna,COAD.TCGA.A6.2680.11A.01R.1757.13_mirna,COAD.TCGA.AA.3525.11A.01R.1757.13_mirna,READ.TCGA.AF.2689.11A.01R.1757.13_mirna,COAD.TCGA.A6.2685.11A.01R.1757.13_mirna,COAD.TCGA.A6.2683.11A.01R.1757.13_mirna,COAD.TCGA.AA.3527.11A.01R.1757.13_mirna,COAD.TCGA.A6.2684.11A.01R.1757.13_mirna,READ.TCGA.AF.2691.11A.01R.1757.13_mirna,GBM.TCGA.06.0675.11A.32R.A36C.13_mirna,...,LIHC.TCGA.DD.A1EE.11A.11R.A130.13_mirnaA,LUAD.TCGA.44.6144.11A.01H.2169.13_mirnaB,BLCA.TCGA.BL.A13J.11A.13R.A10V.13_mirnaB,BRCA.TCGA.BH.A0E1.11A.13R.A090.13_mirnaA,STAD.TCGA.HU.A4GH.11A.11R.A360.13_mirna,PAAD.TCGA.HV.A5A3.11A.11R.A26Y.13_mirna,UCEC.TCGA.FL.A1YQ.11A.11R.A17A.13_mirnaA,LUSC.TCGA.56.7580.11A.01H.2044.13_mirnaA,HNSC.TCGA.CV.7103.11A.01R.2015.13_mirnaB,BRCA.TCGA.BH.A1FB.11A.33R.A13P.13_mirnaA
Codon,,,,,,,,,,,,,,,,,,,,,
TTT,0.713307,0.656352,0.649539,0.695144,0.665063,0.640611,0.676874,0.668205,0.675947,0.625627,...,0.560126,0.658939,0.772357,0.564064,0.745214,0.678691,0.647120,0.607238,0.601936,0.542109
TTC,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
TTA,3.902278,2.333624,2.554340,3.908948,3.199289,3.176690,4.501481,3.868223,3.927721,6.324342,...,4.852128,2.324556,3.298021,2.547575,2.591780,4.430703,6.427682,2.690171,3.124968,2.108453
TTG,2.093227,1.282170,1.422957,1.928169,2.190665,1.771189,1.986257,1.975040,2.273330,2.695048,...,2.004232,1.677396,2.192919,1.357610,1.296808,1.779033,2.119600,1.482835,1.937516,1.321405
TCT,1.512215,0.092782,0.120679,0.139853,1.435318,1.371970,1.432317,0.182548,0.293834,0.082066,...,0.071503,1.231417,1.517047,0.308494,0.224394,0.212184,0.065920,0.115860,0.162530,0.127289


# Get SDAw Codon Weights per TCGA Tissue

In [4]:
# Average SDAw across tissues
# Extract tissue names from column names
tissue_names = SDAw_TCGA.dropna(axis=1).columns.str.split('.').str[0]

# Compute the average for each distinct tissue prefix with geometric mean
average_SDAw = SDAw_TCGA.dropna(axis=1).T.groupby(tissue_names)
average_SDAw = average_SDAw.aggregate(lambda x: np.exp(np.mean(np.log(x))))


In [5]:
average_SDAw

Codon,TTT,TTC,TTA,TTG,TCT,TCC,TCA,TCG,TAT,TAC,...,GCA,GCG,GAT,GAC,GAA,GAG,GGT,GGC,GGA,GGG
BLCA,0.678631,1.000000,2.867081,1.741494,0.545448,0.332788,0.813246,2.676789,0.729802,1.000000,...,1.880292,2.269181,0.618062,1.000000,1.422793,0.998864,0.624955,0.639964,1.242312,1.599514
BRCA,0.560519,1.002934,2.054473,1.234086,0.318913,0.196118,0.870994,3.602979,0.611686,1.000021,...,1.544973,2.782136,0.516253,1.024371,1.051819,0.987949,0.503493,0.595729,1.167651,1.476160
CESC,0.674468,1.000000,2.774683,1.439145,0.198669,0.081841,1.165066,4.063922,0.713520,1.000000,...,1.920234,2.852076,0.598494,1.000000,1.302329,1.000000,0.781920,0.857161,1.196213,1.656235
CHOL,0.517375,1.006524,3.661081,1.593973,0.159093,0.079933,1.035613,2.880704,0.565157,1.000000,...,1.545880,3.660400,0.514258,1.021354,0.936103,0.996473,0.527137,0.526077,0.942647,1.436785
COAD,0.666124,1.000000,3.307638,1.779732,0.577465,0.340428,1.185328,3.074443,0.708370,1.000000,...,1.343906,1.964194,0.638239,1.000000,0.876353,1.000000,0.895223,0.809055,1.284919,1.596567
ESCA,0.688855,1.000000,4.677285,1.816533,0.265821,0.124290,1.426447,5.305361,0.844952,1.000000,...,2.227336,3.225534,0.611510,1.000000,1.369098,1.000000,0.739580,0.628357,1.466081,1.885905
GBM,0.635617,1.000000,6.471916,2.618422,0.135961,0.063666,1.516549,3.478358,0.694792,1.000000,...,1.822269,3.100283,0.603595,1.000000,1.137451,1.000000,1.010843,0.897374,0.895117,1.086139
HNSC,0.619304,1.000000,3.065517,1.778028,0.176481,0.085117,0.840897,3.191854,0.663336,1.001300,...,1.809687,3.241946,0.590781,1.000000,1.391843,0.969589,0.845234,0.912674,0.962690,1.457478
KICH,0.581547,1.000000,4.894961,1.917626,0.310241,0.192563,1.278132,3.143958,0.649755,1.000000,...,1.730772,3.413931,0.551399,1.000444,1.164976,0.985878,0.670703,0.679243,1.072580,1.496294
KIRC,0.546748,1.001026,4.236059,1.663787,0.214837,0.134848,1.357381,3.872549,0.610419,1.000000,...,1.667085,2.737218,0.518505,1.006387,1.206517,0.941943,0.449084,0.486534,1.125101,1.466508


In [6]:
# Save codon SDA weights
average_SDAw.to_csv(os.path.join(data_path, 'SDA', 'SDAw.csv'))

# Calculate SDA for each Transcript

In [185]:
# Calculate SDA for a given sequence for all TCGA tissues
def calculate_SDA(row):
    # get the codon sequence
    codons = [row['seq'][i:i+3] for i in range(0, len(row['seq']), 3)][1:-1]
    # filter codons that are in the average_SDAw index
    valid_codons = [codon for codon in codons if codon in average_SDAw.columns]
    if not valid_codons:
        return np.nan
    # get the codon SDAw values (if they exist)
    SDAw_values = average_SDAw.loc[:, valid_codons]
    # geometric mean
    return np.exp(np.log(SDAw_values).mean(axis=1))

# Apply the function with progress bar and parallelization
tqdm.pandas()
transcript_SDAs = test_with_results.swifter.apply(calculate_SDA, axis=1)

Pandas Apply: 100%|██████████| 15976/15976 [00:37<00:00, 430.08it/s]


In [186]:
# Append transcript_SDAs columns to the test_with_results dataframe
test_with_results = pd.concat([test_with_results, transcript_SDAs], axis=1)
test_with_results

,cluster,length (aa),query_gene,query_transcript,seq,amino_acid_seq,median,subject_transcript,average_pident,gene,...,LUSC,PAAD,PCPG,PRAD,READ,SKCM,STAD,THCA,THYM,UCEC
0,2191,932,ENSG00000134324,ENST00000396097,ATGAGCAGAGTGCAGACCATGAATTACGTGGGGCAGTTAGCCGGCC...,MSRVQTMNYVGQLAGQVFVTVKELYKGLNPATLSGCIDIIVIRQPN...,"(expr_pre75_90, expr_pre50_75, expr_pre75_90, ...",ENST00000261596,48.943442,ENSG00000134324,...,0.678064,0.735033,0.677625,0.752274,0.858980,0.834419,0.757865,0.601648,0.657956,0.684410
1,5329,249,ENSG00000172456,ENST00000634606,ATGTGGCTGGACCATCGAGCAGTCAGTCAAGTTAACAGGATCAATG...,MWLDHRAVSQVNRINETKHSVLQYVGGVMSVEMQAPKLLWLKENLR...,"(expr_pre25_50, expr_pre25_50, expr_pre25_50, ...",NaN,0.000000,ENSG00000172456,...,0.793935,0.874142,0.800600,0.812958,0.947686,0.924045,0.907566,0.718346,0.803052,0.810617
2,7096,463,ENSG00000118495,ENST00000416623,ATGGCCACGTTCCCCTGCCAGTTATGTGGCAAGACGTTCCTCACCC...,MATFPCQLCGKTFLTLEKFTIHNYSHSRERPYKCVQPDCGKAFVSR...,"(expr_pre25_50, expr_pre50_75, expr_pre50_75, ...",ENST00000367405,18.618728,ENSG00000118495,...,0.644232,0.660085,0.593801,0.744197,0.853622,0.819539,0.698442,0.569415,0.596356,0.615900
3,7405,432,ENSG00000257950,ENST00000550383,ATGGGGCAGGCGGGCTGCAAGGGGCTCTGCCTGTCGCTGTTCGACT...,MGQAGCKGLCLSLFDYKTEKYVIAKNKKVGLLYRLLQASILAYLVV...,NaN,ENST00000413302,38.319426,ENSG00000257950,...,0.717878,0.774066,0.713374,0.764118,0.843550,0.835170,0.782738,0.659549,0.714328,0.717513
4,5500,57,ENSG00000107669,ENST00000690706,ATGGCTTTCTGGGCGGGGGGTTCGCCCAGCGTCGTGGACTATTTCC...,MAFWAGGSPSVVDYFPSEDFYRCGYCKNESGSRSNGMWAHSMTVQD...,NaN,NaN,0.000000,ENSG00000107669,...,0.732432,0.794586,0.785613,0.831424,0.851798,0.880568,0.795682,0.725671,0.738497,0.765912
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15971,3560,726,ENSG00000081277,ENST00000367324,ATGAACCACTCGCCGCTCAAGACCGCCTTGGCGTACGAATGCTTCC...,MNHSPLKTALAYECFQDQDNSTLALPSDQKMKTGTSGRQRVQEQVM...,"(expr_pre75_90, expr_pre75_90, expr_pre25_50, ...",ENST00000331563,23.066762,ENSG00000081277,...,0.643007,0.669001,0.628341,0.703783,0.752729,0.744786,0.693795,0.594126,0.624006,0.627310
15972,3464,758,ENSG00000105298,ENST00000429344,ATGGGTCGGGACACACGCTCGCGCTCGCGGTCCGCGGGTCGCCGGG...,MGRDTRSRSRSAGRRGRRRQSQSGSRSRSRSHGRRNRRRREDEGRR...,"(expr_pre75_90, expr_pre75_90, expr_pre75_90, ...",NaN,0.000000,ENSG00000105298,...,0.716367,0.758801,0.715366,0.778625,0.841444,0.850728,0.745238,0.695920,0.703011,0.713429
15973,2613,153,ENSG00000066468,ENST00000683035,ATGGTCAGCTGGGGTCGTTTCATCTGCCTGGTCGTGGTCACCATGG...,MVSWGRFICLVVVTMATLSLARPSFSLVEDTTLEPEEPPTKYQISQ...,NaN,ENST00000393637,21.568418,ENSG00000066468,...,0.738595,0.799676,0.717163,0.769692,0.868629,0.853619,0.803811,0.675980,0.742708,0.726433
15974,530,61,ENSG00000197892,ENST00000522355,ATGGGGGACTCCAAAGTGAAAGTGGCGGTGCGGATACGACCCATGA...,MGDSKVKVAVRIRPMNRRETDLHTKCVVDVDANKVILNPVNTNLSK...,"(expr_low25, expr_low25, expr_pre25_50, expr_l...",ENST00000635823,32.786557,ENSG00000197892,...,0.719477,0.772581,0.721637,0.796867,0.863974,0.855849,0.806456,0.645447,0.696553,0.726112


# Save DataFrame

In [187]:
pd.to_pickle(test_with_results, os.path.join(data_path, '2_test_with_bart30_and_SDA.pkl'))